# Prep ParlaSpeech — chapter 1

Parse a ParlaSpeech-{LANG} JSONL into the canonical pipeline JSONL. Two outputs from one pass:

1. **`parlaspeech_{lang}_instance.jsonl`** — one record per utterance with instance-level labels (`filled_pause_present`).
2. **`parlaspeech_{lang}_frame.jsonl`** — one record per utterance with a **50 Hz frame label sequence** (`labels.filled_pause`).

**Inputs**
- `data/unpacked/ParlaSpeech-{LANG}/ParlaSpeech-{LANG}.v3.0/ParlaSpeech-{LANG}.v3.0.jsonl`
  (decompressed by `10_download_data.ipynb` from `.jsonl.gz`)

**Audio note**
ParlaSpeech v3.0 does not bundle audio. Each JSONL record has an `audio` field with a relative path.
Set `cfg.audio_base_dir` when audio is available. The prep step runs without audio —
label computation uses only timestamps already in the JSONL.

Unlike `11a_prep_ROG-art.ipynb` (which parses EXB tiers), ParlaSpeech ships filled-pause
annotations pre-computed — no tier stitching needed.

---

## 0. Setup

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
if HERE.name != "1_data_prep":
    candidate = HERE / "1_data_prep"
    if candidate.exists():
        HERE = candidate
sys.path.insert(0, str(HERE))

import utils_dataprep as udp
PROJECT_ROOT = udp.PROJECT_ROOT
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

---

## 1. Config

All knobs at the top. Set `lang` to match the dataset you downloaded.
`test_mode=True` caps the stream at `test_n_records` lines for quick checks.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Config:
    # Language variant. One of: HR, RS, PL, CZ
    lang: str = "HR"

    # Path to the decompressed JSONL. Leave "" to auto-detect from lang.
    jsonl_path: str = ""

    # Audio base directory — the root under which the `audio` field paths are relative.
    # ParlaSpeech v3.0 does NOT include audio. Set this when you have it.
    # Leave None to skip audio resolution (output audio_path will be null).
    audio_base_dir: str | None = None

    # Frame rate — hard-locked to 50 Hz (must match chapter 4).
    frame_rate_hz: int = 50

    # Output JSONL paths. Left as "" → auto-set from lang below.
    output_instance_jsonl: str = ""
    output_frame_jsonl:    str = ""

    # Split ratios (train / dev / test), assigned deterministically by speaker.
    split_ratios: tuple = (0.8, 0.1, 0.1)
    split_seed:   int   = 42

    # Filtering
    min_duration_s:     float = 0.1    # drop utterances shorter than this
    skip_failed_pauses: bool  = True   # drop records where filled_pauses is None

    # Test mode
    test_mode:      bool = False       ############ TEST MODE
    test_n_records: int  = 1000

cfg = Config()

# Auto-set paths from lang
_lang = cfg.lang
if not cfg.jsonl_path:
    cfg.jsonl_path = (
        f"data/unpacked/ParlaSpeech-{_lang}/"
        f"ParlaSpeech-{_lang}.v3.0/"
        f"ParlaSpeech-{_lang}.v3.0.jsonl"
    )
if not cfg.output_instance_jsonl:
    cfg.output_instance_jsonl = f"data/processed_jsonl/parlaspeech_{_lang.lower()}_instance.jsonl"
if not cfg.output_frame_jsonl:
    cfg.output_frame_jsonl = f"data/processed_jsonl/parlaspeech_{_lang.lower()}_frame.jsonl"

print(cfg)

---

## 2. Locate the JSONL

In [ ]:
jsonl_path = PROJECT_ROOT / cfg.jsonl_path

if not jsonl_path.exists():
    raise FileNotFoundError(
        f"JSONL not found at {jsonl_path}\n"
        f"Run 10_download_data.ipynb with datasets=['ParlaSpeech-{cfg.lang}'] first, "
        "then unpack, then come back here."
    )

size_mb = jsonl_path.stat().st_size / 1e6
print(f"✅ Found: {jsonl_path.relative_to(PROJECT_ROOT)}")
print(f"   Size:  {size_mb:.1f} MB")

---

## 3. Preflight — peek at records

Print the first few records to verify the schema looks right.
Check: `filled_pauses` should be a list (possibly empty `[]`) or `None` (failed inference).

In [ ]:
import json

print(f"First 3 records from {jsonl_path.name}:\n")
with open(jsonl_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        r = json.loads(line)
        si  = r.get("speaker_info", {})
        fps = r.get("filled_pauses")
        print(f"--- Record {i} ---")
        print(f"  id:           {r['id']}")
        print(f"  audio:        {r.get('audio')}")
        print(f"  audio_length: {r.get('audio_length', 0):.3f}s")
        print(f"  text:         {str(r.get('text',''))[:80]}")
        if fps is None:
            print(f"  filled_pauses: None  (inference failed)")
        else:
            print(f"  filled_pauses: {len(fps)} event(s)  {fps[:2]}")
        print(f"  lang/speaker: {si.get('Lang','?')} / {si.get('Speaker_ID','?')}")
        print()

---

## 4. Stream and parse

Stream the JSONL line-by-line to avoid loading the full (potentially multi-GB) file into memory.
Applies `skip_failed_pauses` and `min_duration_s` filters.

In [ ]:
from tqdm.auto import tqdm

records       = []
n_total       = 0
n_skip_none   = 0
n_skip_short  = 0

with open(jsonl_path, encoding="utf-8") as f:
    for line in tqdm(f, desc=f"parsing ParlaSpeech-{cfg.lang}", unit=" lines"):
        n_total += 1
        if cfg.test_mode and n_total > cfg.test_n_records:
            break

        r   = json.loads(line)
        fps = r.get("filled_pauses")
        dur = r.get("audio_length", 0)

        if cfg.skip_failed_pauses and fps is None:
            n_skip_none += 1
            continue

        if dur < cfg.min_duration_s:
            n_skip_short += 1
            continue

        records.append(r)

print(f"\nRead {n_total} lines")
print(f"  Kept:                      {len(records)}")
print(f"  Skipped (None FPs):        {n_skip_none}")
print(f"  Skipped (too short):       {n_skip_short}")

---

## 5. Stats

Filled-pause prevalence, duration distribution, speaker inventory.

In [ ]:
from collections import Counter
import statistics

fps_counts  = Counter()
durations   = []
speakers    = Counter()

for r in records:
    fps = r.get("filled_pauses") or []
    fps_counts[len(fps)] += 1
    durations.append(r.get("audio_length", 0))
    sp = r.get("speaker_info", {}).get("Speaker_ID", "unknown")
    speakers[sp] += 1

n_pos = sum(v for k, v in fps_counts.items() if k > 0)
total = len(records)

print(f"Records:               {total}")
print(f"  With filled pause:   {n_pos} ({100*n_pos/total:.1f}%)")
print(f"  Without:             {total - n_pos}")
print(f"\nDuration:  mean={statistics.mean(durations):.2f}s  "
      f"median={statistics.median(durations):.2f}s  "
      f"total={sum(durations)/3600:.1f}h")
print(f"\nSpeakers:  {len(speakers)} unique")
print(f"\nFP count distribution (top 10 counts):")
for k, v in sorted(fps_counts.items())[:10]:
    bar = "█" * min(40, int(40 * v / total))
    print(f"  {k:3d} FP(s): {v:6d}  {bar}")

---

## 6. Frame labels

Convert `filled_pauses` interval lists to 50 Hz binary frame sequences.

In [ ]:
def compute_frame_labels(filled_pauses, audio_length, frame_rate_hz):
    """Return a list of 0/1 ints, one per frame at frame_rate_hz."""
    n_frames = round(audio_length * frame_rate_hz)
    labels   = [0] * n_frames
    for fp in (filled_pauses or []):
        f_s = max(0,        round(fp["time_s"] * frame_rate_hz))
        f_e = min(n_frames, round(fp["time_e"] * frame_rate_hz))
        for i in range(f_s, f_e):
            labels[i] = 1
    return labels


# Smoke test on first record
ex = records[0]
seq = compute_frame_labels(
    ex.get("filled_pauses") or [], ex["audio_length"], cfg.frame_rate_hz
)
print(f"Smoke test — {ex['id']}")
print(f"  audio_length = {ex['audio_length']:.3f}s → {len(seq)} frames at {cfg.frame_rate_hz} Hz")
print(f"  FP events:   {ex.get('filled_pauses')}")
print(f"  Frame seq:   {seq[:30]}...")
print(f"  Positive frames: {sum(seq)}/{len(seq)} ({100*sum(seq)/max(1,len(seq)):.1f}%)")

---

## 7. Split assignment

Deterministic split by speaker — same speaker always lands in the same split,
preventing data leakage across train/dev/test.

In [ ]:
import hashlib
from collections import Counter

train_r, dev_r, _ = cfg.split_ratios

def speaker_split(speaker_id: str) -> str:
    h = hashlib.md5(f"{cfg.split_seed}:{speaker_id}".encode()).hexdigest()
    v = int(h, 16) / (16 ** len(h))
    if v < train_r:
        return "train"
    elif v < train_r + dev_r:
        return "dev"
    return "test"

split_counts = Counter()
for r in records:
    sp = r.get("speaker_info", {}).get("Speaker_ID", "unknown")
    r["_split"] = speaker_split(sp)
    split_counts[r["_split"]] += 1

print("Split distribution:")
for split in ["train", "dev", "test"]:
    n = split_counts[split]
    print(f"  {split:5s}: {n:6d} ({100*n/len(records):.1f}%)")

---

## 8. Audio path resolution

⚠️ **ParlaSpeech v3.0 does not include audio.**

Each record has an `audio` field with a relative path
(e.g. `"I27W0FVLD2I/I27W0FVLD2I_27158.08-27159.66.flac"`).

Set `cfg.audio_base_dir` to the root directory where ParlaSpeech audio lives
and re-run this cell. Until then `audio_path` is `null` in the output JSONLs.

All downstream cells work without audio — labels compute from timestamps alone.

In [ ]:
from pathlib import Path

def resolve_audio(record, audio_base_dir):
    if audio_base_dir is None:
        return None
    return str(Path(audio_base_dir) / record["audio"])

n_resolved = 0
for r in records:
    r["_audio_path"] = resolve_audio(r, cfg.audio_base_dir)
    if r["_audio_path"] is not None:
        n_resolved += 1

if cfg.audio_base_dir is None:
    print("⚠️  cfg.audio_base_dir is None — audio_path will be null in output.")
    print("   Set it and re-run this cell when audio is available.")
else:
    print(f"✅ Resolved {n_resolved}/{len(records)} paths under {cfg.audio_base_dir}")

---

## 9. Write instance JSONL

One record per utterance. `labels.filled_pause_present` is 1 if the utterance
contains at least one filled pause, 0 otherwise.

In [ ]:
import json

out_instance = PROJECT_ROOT / cfg.output_instance_jsonl
out_instance.parent.mkdir(parents=True, exist_ok=True)

def make_instance_record(r):
    fps = r.get("filled_pauses") or []
    si  = r.get("speaker_info", {})
    return {
        "file_id":       r["id"],
        "audio_path":    r["_audio_path"],
        "audio_path_raw": r.get("audio"),   # original relative path from JSONL
        "duration_s":    r.get("audio_length"),
        "text":          r.get("text"),
        "lang":          si.get("Lang", cfg.lang),
        "split":         r["_split"],
        "labels": {
            "filled_pause_present": 1 if len(fps) > 0 else 0,
        },
        "meta": {
            "speaker_id":      si.get("Speaker_ID"),
            "speaker_gender":  si.get("Speaker_gender"),
            "speaker_role":    si.get("Speaker_role"),
            "date":            si.get("Date"),
            "party":           si.get("Speaker_party"),
            "n_filled_pauses": len(fps),
        },
    }

with open(out_instance, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(make_instance_record(r), ensure_ascii=False) + "\n")

print(f"✅ Wrote {len(records)} records → {out_instance.relative_to(PROJECT_ROOT)}")

---

## 10. Write frame JSONL

One record per utterance. `labels.filled_pause` is a 50 Hz binary list.
Chapter 4 (`train_frame.ipynb`) consumes this directly.

In [ ]:
import json

out_frame = PROJECT_ROOT / cfg.output_frame_jsonl
out_frame.parent.mkdir(parents=True, exist_ok=True)

def make_frame_record(r):
    fps = r.get("filled_pauses") or []
    dur = r.get("audio_length", 0)
    si  = r.get("speaker_info", {})
    return {
        "file_id":       r["id"],
        "audio_path":    r["_audio_path"],
        "audio_path_raw": r.get("audio"),
        "duration_s":    dur,
        "lang":          si.get("Lang", cfg.lang),
        "split":         r["_split"],
        "labels": {
            "filled_pause": compute_frame_labels(fps, dur, cfg.frame_rate_hz),
        },
        "meta": {
            "speaker_id":     si.get("Speaker_ID"),
            "speaker_gender": si.get("Speaker_gender"),
            "speaker_role":   si.get("Speaker_role"),
            "date":           si.get("Date"),
            "party":          si.get("Speaker_party"),
        },
    }

with open(out_frame, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(make_frame_record(r), ensure_ascii=False) + "\n")

print(f"✅ Wrote {len(records)} records → {out_frame.relative_to(PROJECT_ROOT)}")

---

## 11. Sanity checks

Frame label lengths vs audio duration, and instance label distribution.

In [ ]:
import json

# 11a. Frame label length vs audio_length
mismatches = []
with open(out_frame, encoding="utf-8") as f:
    for line in f:
        r        = json.loads(line)
        dur      = r["duration_s"] or 0
        expected = round(dur * cfg.frame_rate_hz)
        actual   = len(r["labels"]["filled_pause"])
        if abs(actual - expected) > 1:
            mismatches.append((r["file_id"], expected, actual))

if mismatches:
    print(f"⚠️  {len(mismatches)} frame-length mismatch(es):")
    for fid, exp, act in mismatches[:5]:
        print(f"  {fid}: expected ~{exp} got {act}")
else:
    print("✅ All frame label lengths match audio_length (±1 frame)")

# 11b. Instance label distribution
with open(out_instance, encoding="utf-8") as f:
    inst_labels = [json.loads(l)["labels"]["filled_pause_present"] for l in f]
n_pos = sum(inst_labels)
n_tot = len(inst_labels)
print(f"\nInstance label: {n_pos}/{n_tot} positive ({100*n_pos/n_tot:.1f}% filled-pause)")
print(f"               {n_tot - n_pos}/{n_tot} negative")

---

## Next

- **Chapter 2** — point `20_sniff_dataset.ipynb` at the frame JSONL to inspect label distributions.
- **Chapter 4 (frame model)** — load `parlaspeech_{lang}_frame.jsonl`.
  Set `audio_base_dir` before training (cell 8 above).
- Run another language by changing `cfg.lang` and re-running.

**Audio path resolution is the known blocker.** The paths inside the JSONL are relative to the
ParlaSpeech audio archive root, which varies by server. Track progress in BLUEPRINT.md section 11.